# Etapa 2 — Preparação da base

Consome as funções de `datathon_mlet.data_prep` (Etapa 2, já testadas em
`tests/test_data_prep.py`) para transformar o dataset tratado na Etapa 1 em
**contexto**, **ação** e **recompensa** prontos para o baseline e a política
adaptativa da Etapa 3. Este notebook não reimplementa lógica — só orquestra
e inspeciona o resultado.

## Passo 1 — Carregar dataset tratado

`load_clean_dataset` só lê o parquet gerado no final da Etapa 1
(`notebooks/01_eda.ipynb`, passo 8). O pacote `datathon_mlet` já está
instalado em modo editável pelo `uv sync`, então o import funciona sem
truques de `sys.path`.

In [3]:
from pathlib import Path

from datathon_mlet.data_prep import load_clean_dataset, prepare_features

CLEAN_DATASET_PATH = Path("../data/processed/bank_marketing_clean.parquet")

df_clean = load_clean_dataset(CLEAN_DATASET_PATH)
df_clean.shape

(41188, 21)

## Passo 2 — Aplicar `prepare_features`

Uma chamada só separa as 3 responsabilidades — `context` (features do cliente), `action` (braço `contact`), `reward` (`y` binarizado). É o `PreparedDataset` criado e testado na Etapa 2 (`tests/test_data_prep.py`).

In [4]:
prepared = prepare_features(df_clean)

prepared.context.shape, prepared.action.shape, prepared.reward.shape

((41188, 17), (41188,), (41188,))

## Passo 3 — Inspecionar `context`

Confirma visualmente que as exclusões aconteceram certo — não basta confiar no código sem olhar o dado.

In [5]:
prepared.context.columns.tolist()

['age',
 'job',
 'marital',
 'education',
 'default',
 'housing',
 'loan',
 'month',
 'day_of_week',
 'campaign',
 'pdays',
 'previous',
 'poutcome',
 'cons.price.idx',
 'cons.conf.idx',
 'euribor3m',
 'foi_contatado_antes']

In [6]:
prepared.context.head()

,age,job,marital,education,default,housing,loan,month,day_of_week,campaign,pdays,previous,poutcome,cons.price.idx,cons.conf.idx,euribor3m,foi_contatado_antes
0,56,housemaid,married,basic.4y,no,no,no,may,mon,1,999,0,nonexistent,93.994,-36.4,4.857,False
1,57,services,married,high.school,unknown,no,no,may,mon,1,999,0,nonexistent,93.994,-36.4,4.857,False
2,37,services,married,high.school,no,yes,no,may,mon,1,999,0,nonexistent,93.994,-36.4,4.857,False
3,40,admin.,married,basic.6y,no,no,no,may,mon,1,999,0,nonexistent,93.994,-36.4,4.857,False
4,56,services,married,high.school,no,no,yes,may,mon,1,999,0,nonexistent,93.994,-36.4,4.857,False


## Passo 4 — Inspecionar `action` e `reward`

Confirma que `action` tem as categorias certas do braço, e que `reward` bate com `p̂ = 0.1127` já calculado na Etapa 1 — fecha a rastreabilidade EDA → preparação.

In [7]:
prepared.action.value_counts()

contact
cellular     26144
telephone    15044
Name: count, dtype: int64

In [8]:
prepared.reward.value_counts(normalize=True)

y
0    0.887346
1    0.112654
Name: proportion, dtype: float64